In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS ct_oil_gas.sc_metadata 

In [0]:
%sql

DROP table ct_oil_gas.sc_metadata.ingestion_control 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ct_oil_gas.sc_metadata.ingestion_control (
    job_id            INT,
    table_name          STRING,
    source_system       STRING,        
    source_path         STRING,        
    target_layer        STRING, 
    target_catalog      STRING,       
    bronze_schema       STRING,        
    silver_schema       STRING,        
    gold_schema         STRING,        
    active_flag         BOOLEAN,
    load_order          INT,
    watermark_column    STRING,       
    created_at          TIMESTAMP
)

In [0]:
%sql
INSERT INTO ct_oil_gas.sc_metadata.ingestion_control VALUES
-- 1. Customers (SQL Server)
(1, 'oil_gas_data', 'blob', '/Volumes/ct_oil_gas/sc_bronze/v1/raw_data/supply_chain_dataset.xlsx', 'sc_bronze','ct_oil_gas','sc_bronze','sc_silver','sc_gold',1,1,'ingestion_timestamp',current_timestamp())

In [0]:
%sql
SELECT * from ct_oil_gas.sc_metadata.ingestion_control

In [0]:
%sql

-- Table to store column-level transformation rules
CREATE TABLE IF NOT EXISTS config.silver_transformation_config (
  config_id INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
  table STRING NOT NULL,
  column_name STRING NOT NULL,
  transformation_type STRING NOT NULL,  -- CAST, TRIM, UPPER, LOWER, REPLACE, CUSTOM
  transformation_logic STRING,  -- SQL expression or function
  data_type STRING,  -- Target data type for CAST operations
  is_active BOOLEAN DEFAULT TRUE,
);

In [0]:
%sql
drop table ct_oil_gas.sc_metadata.transformation_config

In [0]:
%sql
CREATE TABLE ct_oil_gas.sc_metadata.TransformationConfig
(
    Transformation_id INT NOT NULL,
    Job_Id INT NOT NULL,
    Transformation_Name STRING,
    Transformation_Arguments STRING,
    Column_Name STRING
);

-- =========================================================
-- CAST transformations (Job_Id = 1 -> oil_gas_transactions)
-- =========================================================
INSERT INTO ct_oil_gas.sc_metadata.TransformationConfig VALUES
(1,  1, 'CAST', '{"type": "string"}',       'transaction_id'),
(2,  1, 'CAST', '{"type": "timestamp_ntz"}',  'transaction_date'),
(3,  1, 'CAST', '{"type": "long"}',        'transaction_year'),
(4,  1, 'CAST', '{"type": "long"}',        'transaction_quarter'),
(5,  1, 'CAST', '{"type": "long"}',        'transaction_month'),
(6,  1, 'CAST', '{"type": "string"}',       'product_name'),
(7,  1, 'CAST', '{"type": "string"}',       'product_category'),
(8,  1, 'CAST', '{"type": "string"}',       'quantity_unit'),
(9,  1, 'CAST', '{"type": "string"}',       'supplier_name'),
(10, 1, 'CAST', '{"type": "string"}',       'supplier_country'),
(11, 1, 'CAST', '{"type": "decimal(2,2)"}',   'supplier_reliability_score'),
(12, 1, 'CAST', '{"type": "string"}',       'refinery_name'),
(13, 1, 'CAST', '{"type": "string"}',       'destination_city'),
(14, 1, 'CAST', '{"type": "string"}',       'transportation_mode'),
(15, 1, 'CAST', '{"type": "long"}',        'ordered_quantity'),
(16, 1, 'CAST', '{"type": "decimal(34,14)"}',  'demand_quantity'),
(17, 1, 'CAST', '{"type": "decimal(34,14)"}',  'available_inventory'),
(18, 1, 'CAST', '{"type": "decimal(36,16)"}',  'unit_price_usd'),
(19, 1, 'CAST', '{"type": "decimal(33,13)"}',  'product_cost_usd'),
(20, 1, 'CAST', '{"type": "decimal(34,14)"}',  'transportation_cost_usd'),
(21, 1, 'CAST', '{"type": "decimal(33,13)"}',  'total_cost_usd'),
(22, 1, 'CAST', '{"type": "long"}',        'expected_lead_time_days'),
(23, 1, 'CAST', '{"type": "long"}',        'actual_lead_time_days'),
(24, 1, 'CAST', '{"type": "long"}',        'delay_days'),
(25, 1, 'CAST', '{"type": "long"}',        'is_delayed'),
(26, 1, 'CAST', '{"type": "long"}',        'is_stockout'),
(27, 1, 'CAST', '{"type": "string"}',       'quality_status'),
(28, 1, 'CAST', '{"type": "decimal(35,15)"}',  'quality_score'),
(29, 1, 'CAST', '{"type": "string"}',       'disruption_type'),
(30, 1, 'CAST', '{"type": "string"}',       'delivery_status'),

-- =========================================================
-- DEDUP_KEY (partition key in Column_Name, order-by in Arguments)
-- =========================================================
(32, 1, 'DEDUP_KEY', '{"order_by": "ingestion_timestamp"}', 'transaction_id'),

-- =========================================================
-- TRIM (all string columns, excluding bronze_timestamp)
-- =========================================================
(33, 1, 'TRIM', '{}', 'transaction_id'),
(34, 1, 'TRIM', '{}', 'product_name'),
(35, 1, 'TRIM', '{}', 'product_category'),
(36, 1, 'TRIM', '{}', 'quantity_unit'),
(37, 1, 'TRIM', '{}', 'supplier_name'),
(38, 1, 'TRIM', '{}', 'supplier_country'),
(39, 1, 'TRIM', '{}', 'refinery_name'),
(40, 1, 'TRIM', '{}', 'destination_city'),
(41, 1, 'TRIM', '{}', 'transportation_mode'),
(42, 1, 'TRIM', '{}', 'quality_status'),
(43, 1, 'TRIM', '{}', 'disruption_type'),
(44, 1, 'TRIM', '{}', 'delivery_status'),
(45,  1, 'date_cast', '{}','transaction_date');

In [0]:
%sql

-- Create transformation config table

CREATE TABLE IF NOT EXISTS ct_oil_gas.sc_metadata.transformation_config (
    config_id           BIGINT GENERATED ALWAYS AS IDENTITY,
    table_name           STRING NOT NULL,
    column_name           STRING,              -- NULL for table-level ops (dedup, drop_nulls, drop_duplicates)
    transformation_type   STRING NOT NULL,       -- cast | trim | dedup_by_key | drop_nulls | drop_duplicates
    target_type           STRING,              -- used when transformation_type = 'cast', e.g. 'decimal(18,2)'
    dedup_key             STRING,              -- used when transformation_type = 'dedup_by_key'
    dedup_order_by         STRING,              -- used when transformation_type = 'dedup_by_key'
    sequence_order         INT NOT NULL,          -- order in which to apply transformations
    active_flag           STRING NOT NULL,
    created_ts            TIMESTAMP 
)
USING DELTA


In [0]:
%sql
-- Insert transformation rules for oil_gas_transactions

-- Table-level: dedupe by transaction_id, keep earliest by ingestion_timestamp
INSERT INTO ct_oil_gas.sc_metadata.transformation_config
    (table_name, column_name, transformation_type, target_type, dedup_key, dedup_order_by, sequence_order, active_flag)
VALUES
    ('oil_gas_transactions', NULL, 'dedup_by_key', NULL, 'transaction_id', 'ingestion_timestamp', 1, 'true');

-- Column-level: date cast
INSERT INTO ct_oil_gas.sc_metadata.transformation_config
    (table_name, column_name, transformation_type, target_type, dedup_key, dedup_order_by, sequence_order, active_flag)
VALUES
    ('oil_gas_transactions', 'transaction_date', 'cast', 'date', NULL, NULL, 2, 'true');

-- Column-level: string trims
INSERT INTO ct_oil_gas.sc_metadata.transformation_config
    (table_name, column_name, transformation_type, target_type, dedup_key, dedup_order_by, sequence_order, active_flag)
VALUES
    ('oil_gas_transactions', 'product_name',        'trim', NULL, NULL, NULL, 3,  'true'),
    ('oil_gas_transactions', 'product_category',     'trim', NULL, NULL, NULL, 4,  'true'),
    ('oil_gas_transactions', 'quantity_unit',        'trim', NULL, NULL, NULL, 5,  'true'),
    ('oil_gas_transactions', 'supplier_name',        'trim', NULL, NULL, NULL, 6,  'true'),
    ('oil_gas_transactions', 'supplier_country',      'trim', NULL, NULL, NULL, 7,  'true'),
    ('oil_gas_transactions', 'refinery_name',        'trim', NULL, NULL, NULL, 8,  'true'),
    ('oil_gas_transactions', 'destination_city',      'trim', NULL, NULL, NULL, 9,  'true'),
    ('oil_gas_transactions', 'transportation_mode',    'trim', NULL, NULL, NULL, 10, 'true'),
    ('oil_gas_transactions', 'quality_status',        'trim', NULL, NULL, NULL, 11, 'true'),
    ('oil_gas_transactions', 'delivery_status',       'trim', NULL, NULL, NULL, 12, 'true'),
    ('oil_gas_transactions', 'disruption_type',       'trim', NULL, NULL, NULL, 13, 'true');

-- Column-level: numeric casts
INSERT INTO ct_oil_gas.sc_metadata.transformation_config
    (table_name, column_name, transformation_type, target_type, dedup_key, dedup_order_by, sequence_order, active_flag)
VALUES
    ('oil_gas_transactions', 'ordered_quantity',           'cast', 'long',           NULL, NULL, 14, 'true'),
    ('oil_gas_transactions', 'demand_quantity',            'cast', 'decimal(18,2)',   NULL, NULL, 15, 'true'),
    ('oil_gas_transactions', 'available_inventory',         'cast', 'decimal(18,2)',   NULL, NULL, 16, 'true'),
    ('oil_gas_transactions', 'unit_price_usd',            'cast', 'decimal(18,2)',   NULL, NULL, 17, 'true'),
    ('oil_gas_transactions', 'product_cost_usd',           'cast', 'decimal(18,2)',   NULL, NULL, 18, 'true'),
    ('oil_gas_transactions', 'transportation_cost_usd',       'cast', 'decimal(18,2)',   NULL, NULL, 19, 'true'),
    ('oil_gas_transactions', 'total_cost_usd',            'cast', 'decimal(18,2)',   NULL, NULL, 20, 'true'),
    ('oil_gas_transactions', 'supplier_reliability_score',     'cast', 'decimal(3,2)',    NULL, NULL, 21, 'true'),
    ('oil_gas_transactions', 'quality_score',             'cast', 'decimal(5,2)',    NULL, NULL, 22, 'true');

-- Column-level: boolean casts
INSERT INTO ct_oil_gas.sc_metadata.transformation_config
    (table_name, column_name, transformation_type, target_type, dedup_key, dedup_order_by, sequence_order, active_flag)
VALUES
    ('oil_gas_transactions', 'is_delayed',  'cast', 'boolean', NULL, NULL, 23, 'true'),
    ('oil_gas_transactions', 'is_stockout', 'cast', 'boolean', NULL, NULL, 24, 'true');

In [0]:
%sql
select * FROM ct_oil_gas.sc_metadata.transformation_config

In [0]:
df = spark.table("ct_oil_gas.sc_bronze.oil_gas_transactions")

df.printSchema()

In [0]:
%sql
INSERT INTO ct_oil_gas.sc_metadata.TransformationConfig VALUES
(45,  1, 'date_cast', '{}','transaction_date');

In [0]:
%sql
Select * from ct_oil_gas.sc_metadata.TransformationConfig

In [0]:
%sql
DELETE FROM ct_oil_gas.sc_metadata.TransformationConfig
WHERE Transformation_id IN (
  SELECT Transformation_id FROM (
    SELECT
      Transformation_id,
      row_number() over(partition by Transformation_id order by Transformation_id) as rn
    FROM ct_oil_gas.sc_metadata.TransformationConfig
  )
  WHERE rn > 1
)

